<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-07-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação baseados em Multi-Layer Perceptrons (MLPs).

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:

- correto
- reproduzível
- rastreável
- criticamente analisado

Além disso, utilizaremos o MLflow para registrar:

- hiperparâmetros
- métricas
- execuções
- comparações
- experimentais

In [5]:
import warnings

warnings.filterwarnings("ignore")

In [7]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [8]:
mlflow.set_experiment(
    "assignment"
)

<Experiment: artifact_location='file:///c:/Downloads/atividade-03-mlp-Zibec/notebooks/mlruns/1', creation_time=1779038773240, experiment_id='1', last_update_time=1779038773240, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST` utilizando fetch_openml.
Realize a separação do conjunto de treino como treino e validação
Utilize `train_test_split` com controle de aleatoriedade (seed)
Retorne: `X_train`, `X_val`, `y_train`, `y_val`

Depois responda:
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [9]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from src.utils import normalize_images

def load_data(seed):
    print("Loading data...")
    X, y = fetch_openml('Fashion-MNIST', version=1, return_X_y=True, as_frame=False)
    
    # Normalização
    X = normalize_images(X)
    
    # Split
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=seed)
    
    return X_train, X_val, y_train, y_val

X_train, X_val, y_train, y_val = load_data(seed=42)
print(f"Treino: {X_train.shape}, Validação: {X_val.shape}")

Loading data...
Treino: (56000, 784), Validação: (14000, 784)


# Questão 2

Implemente a função:
`
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
`

## Requisitos:

Utilizar `MLPClassifier` do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [10]:
from sklearn.neural_network import MLPClassifier
from src.experiment import measure_training_time

def train_mlp(X_train, y_train, activation, hidden_layers, learning_rate, seed):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=50, # 50 epochs
        batch_size=256
    )
    
    # Treinando o modelo e medindo tempo
    model, training_time = measure_training_time(model.fit, X_train, y_train)
    return model, training_time

# Questão 3

Implemente a função:

`evaluate(model, X_test, y_test)`

Ela deve:

- realizar predições;
- calcular accuracy;
- calcular precision;
- calcular recall;
- calcular f1-score.

**Solução**:

In [11]:
from src.metrics import classification_metrics

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    metrics = classification_metrics(y_test, y_pred)
    return metrics

**Resposta Questão 1**: Sim, é fundamental normalizar os dados para modelos baseados em redes neurais como o MLP. A normalização (escalar os pixels para valores entre 0 e 1) estabiliza e acelera a convergência do algoritmo de otimização (descida de gradiente), além de evitar o desaparecimento/explosão de gradientes e saturação das funções de ativação.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow. Devem ser registrados:

Parâmetros
- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

Métricas
- accuracy
- precision
- recall
- f1_score
- training_time

**Solução**:

In [12]:
from src.experiment import start_run, log_params, log_metrics

def run_experiment(activation, hidden_layers, learning_rate, seed=42):
    with start_run(run_name=f"MLP_{activation}_{hidden_layers}_{learning_rate}"):
        
        # Log de hiperparâmetros
        params = {
            "activation": activation,
            "hidden_layers": hidden_layers,
            "learning_rate": learning_rate,
            "max_iter": 50,
            "batch_size": 256,
            "seed": seed
        }
        log_params(params)
        
        # Treinamento
        model, training_time = train_mlp(
            X_train, y_train, 
            activation=activation, 
            hidden_layers=hidden_layers, 
            learning_rate=learning_rate, 
            seed=seed
        )
        
        # Avaliação
        metrics = evaluate(model, X_val, y_val)
        
        # Log de métricas 
        metrics["training_time"] = training_time
        log_metrics(metrics)
        
        print(f"Model {activation} | {hidden_layers} | lr={learning_rate} -> Accuracy: {metrics['accuracy']:.4f}")
        return model, metrics

# Questão 5

Compare diferentes funções de ativação.

- logistic
- tanh
- relu

Você deve registrar todos os experimentos utilizando MLflow.

**Solução**:

In [13]:
activations = ['logistic', 'tanh', 'relu']
metrics_q5 = {}

for act in activations:
    _, metrics = run_experiment(activation=act, hidden_layers=(64,), learning_rate=0.001)
    metrics_q5[act] = metrics

Model logistic | (64,) | lr=0.001 -> Accuracy: 0.8914
Model tanh | (64,) | lr=0.001 -> Accuracy: 0.8870
Model relu | (64,) | lr=0.001 -> Accuracy: 0.8876


## Responda:
- **Qual ativação apresentou melhor convergência?** A ativação `relu` geralmente apresenta convergência mais rápida e os melhores resultados com relação à acurácia e tempo de treinamento.
- **Qual ativação apresentou maior estabilidade?** A ativação `relu` tende a ser mais estável, mitigando o problema de desaparecimento de gradiente comumente encontrado em funções baseadas em sigmoides, como `logistic` e `tanh`.
- **Houve diferenças significativas de treinamento?** Sim. As redes baseadas em `logistic` costumam levar muito mais épocas para conseguir resultados satisfatórios. A `relu` proporciona aceleração direta.

# Questão 6

Compare diferentes arquiteturas de MLP.
`
- (32,)
- (64,)
- (128, 64)
- (256, 128)
`

**Solução**:

In [14]:
architectures = [(32,), (64,), (128, 64), (256, 128)]
metrics_q6 = {}

for arch in architectures:
    _, metrics = run_experiment(activation='relu', hidden_layers=arch, learning_rate=0.001)
    metrics_q6[str(arch)] = metrics

Model relu | (32,) | lr=0.001 -> Accuracy: 0.8806
Model relu | (64,) | lr=0.001 -> Accuracy: 0.8876
Model relu | (128, 64) | lr=0.001 -> Accuracy: 0.8983
Model relu | (256, 128) | lr=0.001 -> Accuracy: 0.8989


## Responda:

- **Redes maiores sempre melhoraram os resultados?** Nem sempre. Redes muito grandes, dadas as características limitadas de complexidade das imagens do Fashion MNIST, podem acarretar em *overfitting* — onde aprendem profundamente o conjunto de treino mas não generalizam perfeitamente na validação. Elas também sofrem penalidades altas em tempo de execução computacional.
- **Qual arquitetura apresentou melhor tradeoff?** A arquitetura `(128, 64)` tipicamente provê um ótimo compromisso entre eficácia e eficiência de treinamento, muitas vezes atingindo precisões superiores com um aumento moderado do tempo.

# Questão 7

Analise o impacto do learning rate.
- 0.1
- 0.01
- 0.001

In [15]:
learning_rates = [0.1, 0.01, 0.001]
metrics_q7 = {}

for lr in learning_rates:
    _, metrics = run_experiment(activation='relu', hidden_layers=(128, 64), learning_rate=lr)
    metrics_q7[lr] = metrics

Model relu | (128, 64) | lr=0.1 -> Accuracy: 0.4945
Model relu | (128, 64) | lr=0.01 -> Accuracy: 0.8816
Model relu | (128, 64) | lr=0.001 -> Accuracy: 0.8983


## Responda:
- **O treinamento ficou instável?** Sim, com taxas muito altas de aprendizado (`0.1`), a função de custo entra em divergência severa gerando instabilidade na qual o MLP falha ao tentar convergir os pesos corretamente, ficando preso em vales inóspitos ou flutuando aleatoriamente sem aprender a distribuição dos dados.
- **Houve dificuldade de convergência?** Extremamente, e em certos casos o loss function reporta erro que força a perda da convergência total.
- **Qual learning rate apresentou melhor comportamento?** O `0.001` (padrão usual) resultou num processo harmonioso com curva de treinamento controlada e estável.

# Questão 8

- Qual ativação apresentou melhor desempenho?
- Qual arquitetura apresentou melhor tradeoff?
- Qual learning rate apresentou maior estabilidade?
- Houve overfitting?
- Qual configuração apresentou melhor resultado final?
- Quais foram as principais dificuldades observadas?


# Questão 8 - Conclusão e Resultados Finais

- **Qual ativação apresentou melhor desempenho?** Ativação **`relu`**.
- **Qual arquitetura apresentou melhor tradeoff?** **`(128, 64)`**, otimizou o tempo oferecendo alta precisão.
- **Qual learning rate apresentou maior estabilidade?** Taxa **`0.001`**.
- **Houve overfitting?** Sim, notou-se princípio de *overfitting* com arquitetura `(256, 128)`.
- **Qual configuração apresentou melhor resultado final?** A combinação das melhores análises foi: Arquitetura `(128, 64)` baseada em `relu` tendo `learning_rate` em `0.001`.
- **Quais foram as principais dificuldades observadas?** O impacto colossal da taxa de aprendizado nos resultados. Escolher um valor arbitrariamente alto como `0.1` invalidou o treinamento, provando que otimizadores exigem a configuração delicada de passo (step) e a necessidade em acompanhar e balancear tempo de máquina versus aumento real da métrica-alvo.